# Nanoscale Connectomics — Modular Full Run (Verbose Colab)
Runs the checkpointed five-phase modular pipeline with deliberately verbose output. Scientific logic is unchanged; this is an execution surface for the merged modular runner.

Phases: **discovery → graph → people → enrichment → finalize**. Each phase writes a checkpoint and summary.


In [ ]:
MODE = 'fresh'  # or 'seed_expand'
REPO = 'https://github.com/wrgr/connectomics-survey.git'
WORK_ROOT = '/content/connectomics-survey'
PACKAGE_DIR = f'{WORK_ROOT}/connectomics_deterministic_pipeline'
STATE_DIR = f'{PACKAGE_DIR}/modular_state'
OUTPUT_DIR = f'{PACKAGE_DIR}/outputs'
CONFIG_PATH = f'{PACKAGE_DIR}/config.colab.yaml'
print('Mode:', MODE)


In [ ]:
import os, sys, time, json, shutil, pathlib, subprocess, getpass
from datetime import datetime, timezone
def stamp(): return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
def banner(s): print('\n'+'='*100+'\n['+stamp()+'] '+s+'\n'+'='*100, flush=True)
def run_verbose(cmd, cwd=None):
    banner('COMMAND: '+' '.join(map(str,cmd)))
    t=time.time(); p=subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    rc=p.wait(); print(f'[{stamp()}] rc={rc} elapsed={time.time()-t:.1f}s',flush=True)
    if rc: raise subprocess.CalledProcessError(rc,cmd)
def show_json(path):
    p=pathlib.Path(path); print('\n',path); print(json.dumps(json.loads(p.read_text()),indent=2) if p.exists() else '(missing)')
def inventory(path):
    p=pathlib.Path(path); fs=sorted(x for x in p.rglob('*') if x.is_file()) if p.exists() else []
    print(f'\nInventory {path}: {len(fs)} files')
    for f in fs[:150]: print(f'{f.stat().st_size:>12,}  {f.relative_to(p)}')


## Clone and reconstruct the current repository implementation


In [ ]:
if os.path.exists(WORK_ROOT): shutil.rmtree(WORK_ROOT)
run_verbose(['git','clone',REPO,WORK_ROOT])
run_verbose(['git','-C',WORK_ROOT,'rev-parse','HEAD'])
run_verbose(['bash','bootstrap_bundle.sh'],cwd=WORK_ROOT)
inventory(PACKAGE_DIR)


## Install dependencies and run verbose offline tests


In [ ]:
run_verbose([sys.executable,'-m','pip','install','-r',f'{PACKAGE_DIR}/requirements.txt','pytest'])
run_verbose([sys.executable,'-m','pytest','-vv'],cwd=PACKAGE_DIR)


## Semantic Scholar key
Use a Colab secret named `SEMANTIC_SCHOLAR_API_KEY` if available. The value is never printed.


In [ ]:
key=None
try:
    from google.colab import userdata
    key=userdata.get('SEMANTIC_SCHOLAR_API_KEY')
except Exception: pass
if not key: key=getpass.getpass('Semantic Scholar API key: ').strip()
if not key: raise RuntimeError('No Semantic Scholar API key supplied')
os.environ['SEMANTIC_SCHOLAR_API_KEY']=key
print('API key present:',bool(key),'length:',len(key),'(value hidden)')


## Build run configuration


In [ ]:
import yaml
src=pathlib.Path(PACKAGE_DIR)/('config.example.yaml' if MODE=='fresh' else 'config.seed-expand.yaml')
cfg=yaml.safe_load(src.read_text()); cfg['mode']=MODE; cfg['outdir']='outputs'
pathlib.Path(CONFIG_PATH).write_text(yaml.safe_dump(cfg,sort_keys=False))
print(pathlib.Path(CONFIG_PATH).read_text())


# Run modular phases
Each cell streams all pipeline output and then prints the checkpoint summary and inventories. You can stop after any successful phase and resume with the next one while the runtime/checkpoints remain available.


In [ ]:
PHASES=['discovery','graph','people','enrichment','finalize']
def run_phase(phase):
    banner('PHASE START: '+phase.upper()); t=time.time()
    run_verbose([sys.executable,'run_pipeline_modular.py','--config',CONFIG_PATH,'--state-dir',STATE_DIR,'--phase',phase],cwd=PACKAGE_DIR)
    banner(f'PHASE COMPLETE: {phase.upper()} ({time.time()-t:.1f}s)')
    show_json(f'{STATE_DIR}/{phase}_summary.json')
    inventory(STATE_DIR); inventory(OUTPUT_DIR)


## 1 — Discovery


In [ ]:
run_phase('discovery')


## 2 — Citation graph, screening, deduplication, ranking


In [ ]:
run_phase('graph')


## 3 — People map and author saturation


In [ ]:
run_phase('people')


## 4 — Crossref, NIH, health, training/outreach enrichment


In [ ]:
run_phase('enrichment')


## 5 — Finalize graphs, tables, coverage, manifest


In [ ]:
run_phase('finalize')


## Final audit and ZIP


In [ ]:
inventory(OUTPUT_DIR)
for n in ['coverage_summary.json','manifest.json']:
    if (pathlib.Path(OUTPUT_DIR)/n).exists(): show_json(pathlib.Path(OUTPUT_DIR)/n)
import zipfile
zip_path='/content/connectomics_modular_run_artifacts.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for base in [pathlib.Path(OUTPUT_DIR),pathlib.Path(STATE_DIR)]:
        if base.exists():
            for p in base.rglob('*'):
                if p.is_file(): z.write(p,arcname=str(p.relative_to(PACKAGE_DIR)))
    z.write(CONFIG_PATH,arcname='config.colab.yaml')
print('Created:',zip_path,pathlib.Path(zip_path).stat().st_size,'bytes')


In [ ]:
try:
    from google.colab import files
    files.download('/content/connectomics_modular_run_artifacts.zip')
except Exception as e: print('Download manually from /content/connectomics_modular_run_artifacts.zip',repr(e))
